# **GitHub Repository Mining Script**

Extracts following contents from an specified GitHub repo:
 - Commit history
 - File changes
 - Code churn (added/deleted LOC)
 - Authors and ownership
 - Timestamps

Need to install following libraries:
 - `pydriller` A package for getting GitHub repo data. Checkout docuemntation [here](https://pydriller.readthedocs.io/en/latest/).


Required modules:
 -

Author: DS4SE Lab


In [ ]:
!pip install pydriller

Importing modules

In [ ]:
import pandas as pd
from pydriller import Repository
from collections import defaultdict
import datetime as dt

Configurations

In [ ]:
REPO_URL = "https://github.com/scikit-learn/scikit-learn"   # change to your repo
OUTPUT_CSV = "/content/drive/MyDrive/DS4SE/repo_mining_output.csv"

In [ ]:
records = []
ownership_counter = defaultdict(lambda: defaultdict(int))

Getting the repository content

In [ ]:
print("Mining repository... please wait")

for commit in Repository(REPO_URL, since=dt.datetime(2025, 1, 1, 0, 0, 0)).traverse_commits():

    author = commit.author.name
    timestamp = commit.author_date

    for mod in commit.modified_files:

        added = mod.added_lines
        deleted = mod.deleted_lines
        churn = added + deleted
        filename = mod.filename
        filepath = mod.new_path or mod.old_path

        # ownership tracking
        ownership_counter[filepath][author] += 1

        records.append({
            "commit_hash": commit.hash,
            "author": author,
            "timestamp": timestamp,
            "file_name": filename,
            "file_path": filepath,
            "change_type": mod.change_type.name,
            "added_lines": added,
            "deleted_lines": deleted,
            "code_churn": churn
        })

Mining repository... please wait


Converting data in dataframe

In [ ]:
df = pd.DataFrame(records)
df.head()

,commit_hash,author,timestamp,file_name,file_path,change_type,added_lines,deleted_lines,code_churn
0,619da3419bf6e07c8216f8c952c41a0ea37fcabf,Fabian Pedregosa,2010-01-05 13:32:38+00:00,__init__.py,scikits/__init__.py,ADD,1,0,1
1,fbb3c512ecbffc97825eaaaf925c9af580608fa8,Fabian Pedregosa,2010-01-05 13:33:16+00:00,__init__.py,scikits/learn/__init__.py,ADD,2,0,2
2,fbb3c512ecbffc97825eaaaf925c9af580608fa8,Fabian Pedregosa,2010-01-05 13:33:16+00:00,COPYING,scikits/learn/datasets/oldfaithful/COPYING,ADD,34,0,34
3,fbb3c512ecbffc97825eaaaf925c9af580608fa8,Fabian Pedregosa,2010-01-05 13:33:16+00:00,README,scikits/learn/datasets/oldfaithful/README,ADD,6,0,6
4,fbb3c512ecbffc97825eaaaf925c9af580608fa8,Fabian Pedregosa,2010-01-05 13:33:16+00:00,__init__.py,scikits/learn/datasets/oldfaithful/__init__.py,ADD,8,0,8


Ownership Calculation

In [ ]:
ownership_data = []

for file, authors in ownership_counter.items():
    total = sum(authors.values())
    for author, count in authors.items():
        ownership_ratio = count / total
        ownership_data.append([file, author, count, ownership_ratio])

ownership_df = pd.DataFrame(
    ownership_data,
    columns=["file_path", "author", "num_commits", "ownership_ratio"]
)
ownership_df.head()

,file_path,author,num_commits,ownership_ratio
0,scikits/__init__.py,Fabian Pedregosa,1,0.500000
1,scikits/__init__.py,Lars Buitinck,1,0.500000
2,scikits/learn/__init__.py,Fabian Pedregosa,29,0.508772
3,scikits/learn/__init__.py,Olivier Grisel,3,0.052632
4,scikits/learn/__init__.py,Gael varoquaux,5,0.087719


In [ ]:
ownership_df.head()

,file_path,author,num_commits,ownership_ratio
0,scikits/__init__.py,Fabian Pedregosa,1,0.500000
1,scikits/__init__.py,Lars Buitinck,1,0.500000
2,scikits/learn/__init__.py,Fabian Pedregosa,29,0.508772
3,scikits/learn/__init__.py,Olivier Grisel,3,0.052632
4,scikits/learn/__init__.py,Gael varoquaux,5,0.087719


Writing Results

In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
ownership_df.to_csv("/content/drive/MyDrive/DS4SE/file_ownership.csv", index=False)

print("Done!")
print(f"Commit-level data saved to: {OUTPUT_CSV}")
print("Ownership data saved to: file_ownership.csv")

Done!
Commit-level data saved to: /content/drive/MyDrive/DS4SE/repo_mining_output.csv
Ownership data saved to: file_ownership.csv


DMM

In [ ]:
repo = Repository("https://github.com/avandeursen/dmm-test-repo")

for commit in repo.traverse_commits():
  print("| {} | {} | {} | {} |".format(
          commit.msg,
          commit.dmm_unit_size,
          commit.dmm_unit_complexity,
          commit.dmm_unit_interfacing
          ))

| Commit with one large method | 0.0 | 1.0 | 1.0 |
| Make large larger, add small method | 0.8 | 1.0 | 1.0 |
| Make large smaller, make small smaller | 0.6666666666666666 | 0.0 | 0.0 |
| Make small method a bit larger | 1.0 | 1.0 | 1.0 |
| Modify every line in large method | None | None | None |
| Make large larger, make small smaller | 0.0 | 1.0 | 1.0 |
| Make small smaller | 0.0 | 0.0 | 0.0 |
| Make large smaller | 1.0 | 0.0 | 0.0 |
| Make large smaller, make small larger | 1.0 | None | None |
| Add second file | 0.12 | 1.0 | 1.0 |
| Increase in one, decrease in other file | 0.75 | 1.0 | 1.0 |
| Offer README explaining the repo purpose | None | None | None |
| Release under Apache 2 license | None | None | None |
| Add method with interfacing on-point | None | None | None |
| Increase interfacing to risky | None | None | 0.0 |
| Add method with complexity on-point | 1.0 | 1.0 | 1.0 |
| Increase complexity to risky | 1.0 | 0.0 | 1.0 |
| Add method with unit size on-point | 1.0 | 1.0 |